# Wednesday — Train Your Own Object Recognizer

This is the **core model of the week**: a classifier that looks at an object and says *which one it is*.

- On **Tuesday** you trained a network to recognize handwritten digits. Today is the same idea — but with **your** objects, from the photos you took on Day 1.
- One big difference: instead of training from scratch, we **stand on a model already trained on millions of images** (this is called *transfer learning*). That's why ~50 photos per object is enough to get good results.
- The result is a small model you can test **on your own laptop webcam at home**, and later drop onto the robot arm.

**How the arm uses your model (Friday):** the arm code finds the objects and crops them out of the picture; *your* model looks at each crop and says what it is. At test time we tell it the **target** (e.g. "today's target is the cube") and it picks whichever crop it labels as that. So any of the objects can be the target — no retraining needed.


> **How to run this:** You're on our GPU server through **JupyterHub** in your browser — everything is already installed. Run each cell with **Shift+Enter**. The live webcam test later runs on **your own laptop** (it needs your camera), so we'll note where to switch.

In [ ]:
# You're on our JupyterHub GPU server — these are already installed.
# Only if you run this on your own laptop instead:
# !pip install torch torchvision opencv-python matplotlib pillow


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True   # tolerate slightly-truncated phone uploads

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", DEVICE)

## 1. Your data

Your **Monday photos are already here.** Monday's lab created a `captured_photos/` folder in your **home directory** — one subfolder per object (your shared objects, a `none/` folder, and any of your own you added). This notebook reads that same folder, so there's **nothing to move or rename**. Whether you have 3 objects, 4, or 5, the model sizes itself to match.

```
captured_photos/            (in your home directory)
  cube/       img001.jpg img002.jpg ...
  cylinder/   ...
  panda/      ...
  none/       empty-table + clutter shots
  <your own>/ ...
```

The cell below splits each object's photos ~80% `train` / 20% `val` (val = photos the model never trains on, so we can check it isn't cheating). Aim for **40-60 photos per object**.

In [ ]:
import os, shutil, random

def split_train_val(source_dir, out_dir, val_fraction=0.2, seed=0):
    """source_dir has one subfolder per class. Copies images into out_dir/train and out_dir/val.
    A class with no photos yet (e.g. an empty none/) is skipped so it can't break loading."""
    random.seed(seed)
    for cls in sorted(os.listdir(source_dir)):
        cls_path = os.path.join(source_dir, cls)
        if not os.path.isdir(cls_path):
            continue
        imgs = [f for f in os.listdir(cls_path)
                if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        if not imgs:
            print(f"  (skipping '{cls}': no photos yet)")
            continue
        random.shuffle(imgs)
        n_val = max(1, int(len(imgs) * val_fraction))
        for split, files in [("val", imgs[:n_val]), ("train", imgs[n_val:])]:
            dst = os.path.join(out_dir, split, cls)
            os.makedirs(dst, exist_ok=True)
            for f in files:
                shutil.copy(os.path.join(cls_path, f), os.path.join(dst, f))
    print("Split complete ->", out_dir)

### 🔧 Your turn — load your Monday photos

The next cell already points at `~/captured_photos` (the folder Monday's lab made), so usually you can **just run it**.

**Do this:**
1. Run the cell.
2. Read the printout — each object should show a healthy count. (The next cell double-checks balance for you.)
3. Only change `SOURCE_FOLDER` if you deliberately put your photos somewhere else.

<details><summary>Hint</summary>

If a class shows **0 photos** or is missing, open the file browser (left panel), go to your **home** folder, and check `captured_photos/` — each object needs its own subfolder with the images actually inside it. An empty folder (like `none/` before you shoot it) is skipped automatically.
</details>

In [ ]:
# Your Monday photos live in ~/captured_photos (your home directory) — this already points there.
import os
SOURCE_FOLDER = os.path.expanduser("~/captured_photos")   # created by Monday's lab

# Split 80% train / 20% val into my_objects/.  (Re-running is safe; it just re-copies.)
split_train_val(SOURCE_FOLDER, "my_objects", val_fraction=0.2)

# Sanity check: how many photos landed in each class?
for split in ["train", "val"]:
    print(f"\n{split}/")
    root = os.path.join("my_objects", split)
    if os.path.isdir(root):
        for cls in sorted(os.listdir(root)):
            n = len(os.listdir(os.path.join(root, cls)))
            print(f"  {cls:12s} {n} photos")

### Clean incomplete or corrupt photos

Phone uploads occasionally arrive **truncated** (a partial file that shows a gray band) or **corrupt**. This cell drops those from the training copy so they never reach your preview or training. It only touches `my_objects/` — your **originals in `~/captured_photos` stay safe**, so you can re-shoot or re-upload anytime.

In [ ]:
# Drop any photo that didn't upload completely (truncated) or won't open (corrupt).
# This only cleans the training COPY in my_objects/ — your originals in ~/captured_photos stay untouched.
from PIL import Image, ImageFile
import os

removed = []
ImageFile.LOAD_TRUNCATED_IMAGES = False          # strict: a partial upload should fail here
for split in ["train", "val"]:
    for dirpath, _, files in os.walk(os.path.join("my_objects", split)):
        for fn in files:
            if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                p = os.path.join(dirpath, fn)
                try:
                    with Image.open(p) as im:
                        im.load()                # fully decode; raises if truncated/corrupt
                except Exception:
                    removed.append(p)
ImageFile.LOAD_TRUNCATED_IMAGES = True           # back to tolerant, as a safety net

for p in removed:
    os.remove(p)

print(f"Removed {len(removed)} incomplete/corrupt photo(s) from the training copy.")
print("(Your originals in ~/captured_photos are untouched.)")
for p in removed[:10]:
    print("  removed:", os.path.relpath(p, "my_objects"))
if removed:
    print("If a class is now thin, re-shoot and re-upload it.")

### What did the notebook find?

Before training, let's confirm your photos loaded and see how balanced they are. This works for **any** number of objects — 3, 4, or 5.

In [ ]:
# A friendly summary of what's in your dataset (works for any number of classes).
import os

def summarize(root="my_objects"):
    counts = {}
    for split in ("train", "val"):
        d = os.path.join(root, split)
        if not os.path.isdir(d):
            continue
        for cls in sorted(os.listdir(d)):
            p = os.path.join(d, cls)
            if os.path.isdir(p):
                n = len([f for f in os.listdir(p)
                         if f.lower().endswith((".jpg", ".jpeg", ".png"))])
                counts[cls] = counts.get(cls, 0) + n
    return counts

counts = summarize()
objs = [c for c in counts if c != "none"]

if not counts:
    print("No classes found. Check SOURCE_FOLDER above (one subfolder per object) and re-run the split cell.")
else:
    print(f"Found {len(objs)} object class(es): " + (", ".join(objs) or "(none yet)"))
    print("Plus a 'none' class — nice, that keeps the arm from grabbing thin air."
          if "none" in counts else
          "No 'none' class yet — you'll add one near the end (needed for Friday).")
    print()
    for cls, n in sorted(counts.items(), key=lambda kv: -kv[1]):
        flag = "   <-- a bit few; aim for 40-60" if n < 25 else ""
        print(f"  {cls:14s} {n:3d} photos{flag}")

    vals = [counts[c] for c in objs] or [0]
    if vals and max(vals) > 2 * max(1, min(vals)):
        print("\n⚠️  Classes are imbalanced (some have many more photos than others).")
        print("    The model may favor the big class — add photos to the small ones if you can.")
    if len(objs) >= 4:
        print(f"\n🎉 You brought extra objects ({len(objs)} total)! More classes = a better Friday demo.")
    elif len(objs) == 3:
        print("\nThree objects is exactly enough. Shot extras at home? Drop them in and re-run to add them.")


> **💾 Sharing one GPU.** The whole class trains on a single 8 GB GPU today, so memory is tight when everyone runs at once. Two rules of thumb:
> - Keep `BATCH_SIZE` at **16**. If you ever see a `CUDA out of memory` error, drop it to **8** and re-run — smaller batches use less memory (training is just a little slower).
> - Start your training run when the instructor gives the go-ahead for your group, so we don't all hit the GPU in the same second.
>
> If the GPU is jammed, the instructor may move you to the Colab backup — the exact same notebook, your own GPU.


In [ ]:
DATA_DIR = "my_objects"   # <-- folder containing train/ and val/
IMG_SIZE = 224            # the input size the pretrained model expects
BATCH_SIZE = 16
EPOCHS = 8
LEARNING_RATE = 1e-3

# ImageNet normalization stats (the pretrained model was trained with these).
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

# Training images get random changes (augmentation) so the model generalizes.
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])
# Validation/test images are just resized + normalized (no random changes).
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])


In [ ]:
from PIL import Image, ImageOps

def load_upright(path):
    # Respect a phone's rotation tag so portrait photos aren't loaded sideways.
    return ImageOps.exif_transpose(Image.open(path).convert("RGB"))

train_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), train_transform, loader=load_upright)
val_ds   = datasets.ImageFolder(os.path.join(DATA_DIR, "val"),   val_transform,   loader=load_upright)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

CLASS_NAMES = train_ds.classes            # e.g. ['cube', 'cylinder', 'none', 'panda']
NUM_CLASSES = len(CLASS_NAMES)
print("Objects the model will learn:", CLASS_NAMES)
print(f"{len(train_ds)} training images, {len(val_ds)} validation images")

## 2. Look at your data first

Always eyeball a few images before training — it catches mislabeled folders and bad photos early.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import os, random

# Show your REAL photos, upright and in true color — no augmentation here, so you see the data as it is.
# (During training the pipeline randomly flips/rotates them on purpose; that is not shown here.)
samples = []
for cls in CLASS_NAMES:
    folder = os.path.join("my_objects", "train", cls)
    files = [f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    for f in random.sample(files, min(2, len(files))):
        samples.append((cls, os.path.join(folder, f)))
random.shuffle(samples)

plt.figure(figsize=(11, 5))
for i, (cls, path) in enumerate(samples[:8]):
    img = ImageOps.exif_transpose(Image.open(path).convert("RGB"))
    plt.subplot(2, 4, i + 1)
    plt.imshow(img)
    plt.title(cls, fontsize=10)
    plt.axis("off")
plt.tight_layout(); plt.show()

### 🔧 Your turn — be a data detective

The grid shows your photos **as they are** (upright, true color; incomplete uploads were already removed above). Look hard:
- Is every photo in the folder its title says it is? (A `cylinder` photo that slipped into `cube/` will quietly wreck accuracy.)
- Any photo that's blurry, cropped weird, or so dark you can't tell what it is?

If you spot a bad one, delete it and re-run the two cells that build `train_ds`/`val_ds` and this preview. **Fixing data now is the single highest-value thing you'll do all day** — Friday's success rides on it.

> Heads-up: during **training** the model also sees these photos randomly flipped and slightly rotated (that's **augmentation**, on purpose — the black corners are normal). This preview hides that so you can judge your actual data.

<details><summary>Hint</summary>

To remove a bad photo: open the `my_objects/` folder in the JupyterHub file browser (left panel), delete the file, then re-run the two dataset cells (§1) and this preview. Delete the original in `~/captured_photos/` too, so a re-split doesn't bring it back.
</details>

## 3. Build the model (transfer learning)

We take **MobileNetV2**, already trained on millions of images, **freeze** everything it learned, and replace only its final layer with a fresh one sized for *your* objects. We then train just that last layer. It's fast, works on small datasets, and still runs quickly on a Raspberry Pi.


In [ ]:
model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)

# Freeze the pretrained feature extractor (we won't change what it already knows).
for param in model.features.parameters():
    param.requires_grad = False

# Replace the classifier head with one for OUR number of objects.
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
model = model.to(DEVICE)

trainable = [p for p in model.parameters() if p.requires_grad]
print(f"Training {sum(p.numel() for p in trainable):,} parameters (just the new head).")


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(trainable, lr=LEARNING_RATE)


def run_epoch(loader, train):
    model.train() if train else model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    torch.set_grad_enabled(train)
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        if train:
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return loss_sum / total, correct / total


## 4. Train

Watch the **validation accuracy**. If training accuracy climbs but validation stalls or drops, the model is memorizing (overfitting) — collect more/varied photos or add augmentation.


In [ ]:
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    print(f"epoch {epoch:2d}  train acc {tr_acc:5.1%}   val acc {va_acc:5.1%}")


### 🔧 Your turn — read the curve

Look at the numbers that just printed.
- Is **val acc** climbing along with train acc? Good — it's really learning.
- Is train acc high (say 95%+) while val acc is stuck lower? That's **overfitting** — the model is memorizing your exact photos instead of learning the objects.

**Try one experiment:** change `EPOCHS` (cell above the loop) from 8 to 15 and re-run the model-build + training cells. Does val acc keep improving, or does it flatten out? Note the epoch where it stops helping — that's roughly how long *your* data is worth training for.

<details><summary>Hint</summary>

Overfitting looks like a **growing gap**: train accuracy keeps rising while val accuracy flattens. When you see it, *more and more varied photos* help far more than more epochs. "Early stopping" just means: stop at the epoch where val accuracy stopped improving.
</details>


## 5. Check it per object

Overall accuracy can hide a class the model is bad at. Look at accuracy for each object.


In [ ]:
from collections import defaultdict

correct = defaultdict(int); total = defaultdict(int)
model.eval(); torch.set_grad_enabled(False)
for imgs, labels in val_loader:
    preds = model(imgs.to(DEVICE)).argmax(1).cpu()
    for p, y in zip(preds, labels):
        total[CLASS_NAMES[y]] += 1
        correct[CLASS_NAMES[y]] += int(p == y)
for cls in CLASS_NAMES:
    print(f"{cls:12s} {correct[cls]}/{total[cls]} correct")


### 🔧 Your turn — find the weak class

One line above will have the worst score. That's the object your model struggles with most.

Ask yourself *why*: too few photos? All from one angle? Easy to confuse with another object? Write down which class it is — you'll fix it in the last lab block by adding better photos of exactly that object and retraining.

<details><summary>Hint</summary>

Don't guess *why* yet — the **confusion matrix** in the next section shows exactly *what* your weak class gets mistaken for, which tells you which photos to add.
</details>


## 5b. Confusion matrix — what gets mixed up

Per-class accuracy tells you *which* object is weak. A **confusion matrix** tells you *what it's mistaken for* — the fastest way to know which photos to add.

In [ ]:
# Confusion matrix on the validation set (works for any number of classes).
import torch

model.eval()
n = len(CLASS_NAMES)
conf = [[0] * n for _ in range(n)]
with torch.no_grad():
    for imgs, labels in val_loader:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu().tolist()
        for true_i, pred_i in zip(labels.tolist(), preds):
            conf[true_i][pred_i] += 1

# Print a labeled table: rows = the true object, columns = the model's guess.
w = max(10, max(len(c) for c in CLASS_NAMES) + 1)
print("rows = actual object,  columns = the model's guess\n")
print(" " * w + "".join(f"{c[:7]:>8s}" for c in CLASS_NAMES))
for i, c in enumerate(CLASS_NAMES):
    row = "".join(f"{conf[i][j]:>8d}" for j in range(n))
    print(f"{c:<{w}s}{row}")


### 🔧 Your turn — read the confusion matrix

The **diagonal** (top-left to bottom-right) is where the model got it right. Every other number is a mix-up: **row = the true object, column = the model's guess.**

- Which two objects get confused most (the biggest off-diagonal number)?
- Does that match the weak class you found above?

That biggest off-diagonal number is your action item: **take more photos of that pair**, especially from the angles where they look alike, then retrain.

<details><summary>Hint</summary>

If the `cube` row has a big number under the `cylinder` column, the model is calling cubes "cylinder." Add cube photos from angles where its flat faces and corners are clear — the features that tell them apart.
</details>

## 6. Recognize one image + "is this the target?"

`predict` returns the object name and how confident the model is. `is_target` answers the exact question the arm asks: *is what I'm looking at today's target?* The `threshold` lets the model say "not sure / not the target" instead of always guessing.


In [ ]:
from PIL import Image
import torch.nn.functional as F

def predict(pil_image):
    x = val_transform(pil_image).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        probs = F.softmax(model(x)[0], dim=0).cpu()
    idx = int(probs.argmax())
    return CLASS_NAMES[idx], float(probs[idx]), probs

def is_target(pil_image, target_name, threshold=0.6):
    label, conf, _ = predict(pil_image)
    return (label == target_name and conf >= threshold), label, conf

# Example (set a real path):
# label, conf, _ = predict(Image.open("test.jpg").convert("RGB"))
# print(f"I think this is a {label} ({conf:.0%} sure)")


## 7. Test it on brand-new photos (everyone does this)

The real test isn't the val set — it's photos the model has *never seen in any form*. This works for everyone, no webcam needed.

**Do this:**
1. With your phone, take **3–5 fresh photos**: a couple of your objects, and one of something random / an empty table (to test the "nothing here" idea).
2. Upload them into JupyterHub (drag them into the file browser on the left) into a folder called `new_tests/`.
3. Fill in the target below and run the cell.


In [ ]:
# TODO: pick which object is "today's target" (must be one of CLASS_NAMES).
TARGET = CLASS_NAMES[0]        # <-- change me, e.g. "cube"

import os
from PIL import Image

test_folder = "new_tests"
for f in sorted(os.listdir(test_folder)):
    if not f.lower().endswith((".jpg", ".jpeg", ".png")):
        continue
    img = Image.open(os.path.join(test_folder, f)).convert("RGB")
    hit, label, conf = is_target(img, TARGET, threshold=0.6)
    mark = "  <-- TARGET!" if hit else ""
    print(f"{f:20s} -> {label:12s} {conf:.0%}{mark}")


### 🔧 Your turn — tune the threshold

Run the cell above with `threshold=0.6`, then try `0.4` and `0.85` (change it in the `is_target(...)` call).
- Too **low** and it will call random junk your target.
- Too **high** and it will refuse to commit even when it's right.

Find a value where it says "TARGET!" for real targets but stays quiet on the empty-table / random photo. That number is what you'll hand the arm on Friday.

<details><summary>Hint</summary>

A good threshold usually lands between **0.5 and 0.8**. Pick the *lowest* value that still rejects your `none`/empty-table photo — lower thresholds catch more real targets while staying safe.
</details>


## 8. Classical vs. learned — race them 🏁

Before neural nets, a common trick was **"guess by color"**: measure the average color of each object, then label a new photo by whichever object's color is closest. It's fast and needs no training — but it only knows color, not shape or texture.

Let's build that simple color classifier and race it against your neural net on the same val photos.


In [ ]:
import numpy as np
import os
from PIL import Image

def mean_color(pil_image):
    arr = np.asarray(pil_image.convert("RGB").resize((64, 64)), dtype=float)
    return arr.reshape(-1, 3).mean(axis=0)   # average [R, G, B]

# Learn one average color per class from the TRAINING photos.
class_colors = {}
train_root = os.path.join(DATA_DIR, "train")
for cls in CLASS_NAMES:
    folder = os.path.join(train_root, cls)
    colors = [mean_color(Image.open(os.path.join(folder, f)))
              for f in os.listdir(folder)
              if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    class_colors[cls] = np.mean(colors, axis=0)

def color_guess(pil_image):
    c = mean_color(pil_image)
    return min(CLASS_NAMES, key=lambda k: np.linalg.norm(class_colors[k] - c))


In [ ]:
# Race both classifiers on the validation photos.
val_root = os.path.join(DATA_DIR, "val")
color_correct = net_correct = n = 0
for cls in CLASS_NAMES:
    folder = os.path.join(val_root, cls)
    for f in os.listdir(folder):
        if not f.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        img = Image.open(os.path.join(folder, f)).convert("RGB")
        n += 1
        color_correct += int(color_guess(img) == cls)
        net_label, _, _ = predict(img)
        net_correct += int(net_label == cls)

print(f"Color baseline : {color_correct}/{n} = {color_correct/n:.0%}")
print(f"Your neural net: {net_correct}/{n} = {net_correct/n:.0%}")


### 🔧 Your turn — break the color trick

The color classifier does surprisingly OK when your objects are different colors. **Make it fail:**
- Take (or find) two objects that are the *same* color but different shapes. Does the color baseline mix them up while your net keeps them straight?
- Or photograph one object on a busy, colorful background. Whose accuracy drops?

That gap — where color breaks and your net holds — is exactly *why* we train a real model instead of writing color rules by hand.

<details><summary>Hint</summary>

A **cube and a cylinder in the same color** are a classic trap for the color baseline — your neural net should still tell them apart by shape.
</details>


## 9. (Optional, at home) Live webcam test

If you're on your own laptop with a camera and have Python set up locally, you can watch it label objects in real time. On the browser JupyterHub this won't open a camera window — that's expected; the phone-photo test above is the version everyone runs. Hold objects up, press **q** to quit.


In [ ]:
import cv2

def webcam_demo(target_name, threshold=0.6, camera_index=0):
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        print("No webcam found. Use predict() on saved images instead.")
        return
    print("Press 'q' to quit.")
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        label, conf, _ = predict(pil)
        hit = (label == target_name and conf >= threshold)
        color = (0, 200, 0) if hit else (0, 0, 255)
        text = f"{label} {conf:.0%}" + ("  <-- TARGET!" if hit else "")
        cv2.putText(frame, text, (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
        cv2.putText(frame, f"target: {target_name}", (10, 75),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.imshow("Object recognizer (press q to quit)", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release(); cv2.destroyAllWindows()

# Example (pick one of YOUR class names as the target):
# webcam_demo(target_name=CLASS_NAMES[0])


### 🔧 Your turn (important for the arm) — fill your `none` class

For Friday, you don't want the arm grabbing when the target isn't on the table. The fix is the **`none`** class: photos of the empty table plus clutter, hands, and *other* objects. Monday already made the `none/` folder — it just needs photos in it.

**Do this:** check the counts above. If `none` is missing or thin, shoot ~30-50 on your phone (empty desk, your hand, random junk, a couple of non-target objects), upload them into `~/captured_photos/none/`, then re-run from the **split** cell (§1). Afterward, point `is_target` at a `none` photo — it should return `False`. This is what makes "not-recognized" reliable instead of the model guessing an object every time.

<details><summary>Hint</summary>

No `none` photos yet? On this remote day, phone shots of your own desk work great: empty surface, your hand in frame, a spoon or bottle that *isn't* one of your targets. Variety here is what teaches the model "nothing to grab."
</details>

### 🔧 Your turn — add more objects

Monday you may have added 1-2 of your own objects — if so, they're **already in `captured_photos/` and loaded automatically** (check the class list above). Took more photos at home since?

1. Make a new subfolder in `~/captured_photos/` for each new object (e.g. `~/captured_photos/keys/`).
2. Upload its photos, then re-run the **split** cell (§1) and everything from **datasets → training**.
3. Watch the class list grow — the model **resizes its head automatically**, no code changes.

More classes make a more impressive Friday demo. Just the shared three? That's plenty.

<details><summary>Hint</summary>

Give each new object **40-60 photos** too, from varied angles and lighting — same rules as Monday. A class with only 5 photos will drag your whole accuracy down.
</details>

## 9b. Stretch — fine-tune the last block

So far you trained only the new head and kept the backbone **frozen**. **Fine-tuning** unfreezes the last block too and nudges it with a *small* learning rate, so the borrowed features adapt a little to *your* objects. It can add a few points of accuracy — or overfit if you push it.

Run this **only after** your head is trained and you've read the curves. It updates `model` in place, so if it helps, the model you save at the end is the improved one.

In [ ]:
# Unfreeze the last block of the backbone and fine-tune gently.
for p in model.features[-1].parameters():
    p.requires_grad = True

# A NEW optimizer over everything now trainable, with a SMALLER learning rate.
ft_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(ft_params, lr=1e-4)   # 10x smaller than the head's
print(f"Fine-tuning {sum(p.numel() for p in ft_params):,} parameters (head + last block).")

for epoch in range(1, 4):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    print(f"ft-epoch {epoch}: train {tr_acc:.1%} | val {va_acc:.1%}")


If val accuracy went **up**, keep it — the saved model will include the improvement. If it went **down**, you fine-tuned too hard: re-run the **model-build cells (§3)** and the **training loop (§4)** to reset to the frozen-backbone version, then move on.

<details><summary>Why a smaller learning rate?</summary>

The backbone already knows how to see. A big learning rate would smash those good features; `1e-4` (10× smaller than the head's) nudges them gently instead.
</details>

> **⚠️ Before you leave today:** everyone must produce a `recognizer.onnx` file — **Thursday's edge lab starts from it.** Run the save/export cell below and confirm you see `Saved recognizer.pt and recognizer.onnx`. Also write down your **class order** (printed at the end) — you need it to read the model's predictions later.


## 10. Save your model & export ONNX (Thursday needs this)

Save the weights, and export to **ONNX** — the portable format Thursday's lab will shrink and speed up for the Raspberry Pi and the arm.


In [ ]:
# Save the PyTorch weights + the class order (you need the names to read predictions).
torch.save({"state_dict": model.state_dict(), "classes": CLASS_NAMES}, "recognizer.pt")

# Export to ONNX for the edge.
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
torch.onnx.export(model, dummy, "recognizer.onnx",
                  input_names=["image"], output_names=["scores"],
                  dynamic_axes={"image": {0: "batch"}, "scores": {0: "batch"}})
print("Saved recognizer.pt and recognizer.onnx")
print("Class order (remember this!):", CLASS_NAMES)


## Challenges (if you finish early)

1. **Swap the backbone.** Try `resnet18(weights=...)` instead of MobileNetV2 — is it more accurate? Slower?
2. **Fine-tune deeper.** Unfreeze the last block of `model.features` and train with a smaller learning rate (e.g. `1e-4`). Does accuracy improve?
3. **How little data works?** Retrain with only 10 photos per class. Where does accuracy fall off?
4. **Add the `none` class** and see how reliably it rejects objects it shouldn't grab.
5. **Confuse it on purpose.** Find lighting or angles where it fails — then add photos like those and retrain. This is exactly how real datasets get better.

6. **Augmentation on/off.** Retrain using `val_transform` (no random changes) for the training set too. Does val accuracy drop? By how much? That gap is what augmentation bought you.
